# Data Cleaning and Sales Analysis

Explore the sample sales data, apply the reusable cleaning functions, and generate the project visualizations. Run cells from top to bottom.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import business_insights, key_metrics, regional_performance
from src.data_cleaning import cleaning_audit, clean_sales_data, missing_value_summary
from src.visualization import (
    plot_correlation_heatmap, plot_dashboard, plot_missing_values,
    plot_outliers, plot_sales_distribution,
)

DATA_PATH = ROOT / 'data' / 'sales_data.csv'
IMAGE_DIR = ROOT / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

## Inspect the raw data

Review the schema and missing-value counts before making changes. The missingness plot is based on the original input.

In [3]:
raw = pd.read_csv(DATA_PATH)
print(f'Raw rows: {len(raw)}')
display(raw.head())
display(raw.dtypes.to_frame('dtype'))
display(missing_value_summary(raw).to_frame('missing_count'))
plot_missing_values(raw, IMAGE_DIR)

Raw rows: 50


,order_id,order_date,region,product,quantity,unit_price
0,1001,2025-01-03,North,Widget A,2.0,21.0
1,1002,2025-01-10,South,Widget B,1.0,32.5
2,1003,2025-01-15,East,Widget C,3.0,48.0
3,1004,2025-01-20,West,Widget A,4.0,19.5
4,1005,2025-01-25,north,Widget B,2.0,31.5


,dtype
order_id,int64
order_date,str
region,str
product,str
quantity,float64
unit_price,float64


,missing_count
quantity,2


PosixPath('/workspaces/data-cleaning-visualization-analysis/images/missing_values.png')

## Clean and validate

Duplicate order IDs are removed, labels and types are normalized, missing values are imputed, and sales are calculated. IQR outliers are retained and marked for review.

In [4]:
sales = clean_sales_data(raw)
audit = cleaning_audit(raw, sales)
baseline = sales.loc[~sales['is_sales_outlier']]
sales.to_csv(DATA_PATH.with_name('cleaned_sales_data.csv'), index=False)

print(f'Clean rows: {len(sales)}')
print(f'Flagged sales outliers: {int(sales["is_sales_outlier"].sum())}')
display(pd.Series(audit, dtype=object).to_frame('value'))
display(pd.Series(key_metrics(baseline)).to_frame('inlier value'))
display(regional_performance(baseline).round(2))
display(baseline.groupby('product', observed=True)['sales'].sum().sort_values(ascending=False).to_frame('sales'))
for insight in business_insights(baseline, sales):
    print(f'- {insight}')

Clean rows: 49
Flagged sales outliers: 2


,value
raw_rows,50
clean_rows,49
duplicate_order_ids_removed,1
missing_order_ids_removed,0
invalid_dates_removed,0
missing_before,"{'order_id': 0, 'order_date': 0, 'region': 0, ..."
imputed,"{'quantity': 2, 'unit_price': 0, 'region': 0, ..."
outliers_flagged,2


,inlier value
sales,4278.000000
orders,47.000000
average_order,91.021277
median_order,96.000000


,sales,orders,average_order,sales_share
region,,,,
East,1233.0,12,102.75,0.29
North,1131.0,12,94.25,0.26
West,986.0,12,82.17,0.23
South,928.0,11,84.36,0.22


,sales
product,
Widget C,1797.0
Widget B,1519.5
Widget A,961.5


- East leads the inlier comparison at $1,233.00 across 12 orders (29% of inlier sales).
- Widget C contributes the most inlier sales at $1,797.00 (42% of the inlier total).
- 2 IQR-flagged orders contribute $3,340.00 (43.8% of all selected sales); validate these transactions before using headline totals.


## Visualize the cleaned data

The figures are saved in the repository's `images/` directory. The dashboard combines regional, product, order-value, and monthly views.

In [ ]:
figure_paths = [
    plot_outliers(sales, IMAGE_DIR),
    plot_correlation_heatmap(sales, IMAGE_DIR),
    plot_sales_distribution(sales, IMAGE_DIR),
    plot_dashboard(sales, IMAGE_DIR),
]
[str(path.relative_to(ROOT)) for path in figure_paths]